# KDAL Research Training — Exact-09:00-Local HRRR V1

**Status: isolated research experiment; not production and not deployed.** This
generator-backed notebook retains the active KDAL V20 no-peak/full-refit point
and pure-ordinal lineage. Its sole provider change is replacement of HRRR with
the completed exact-09:00 `America/Chicago` candidate cache. The 11:00-local
observation cutoff and 11:15-local prediction/decision contract are unchanged.
All outputs are confined to
`data/calibration/experiments/kdal_hrrr_9am_v1/`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "research_kdal_hrrr_9am_v1_no_peak_stack"
EXPORT_MODEL_WEIGHTS = True
EXPORT_LIVE_MODEL_WEIGHTS = False
POINT_EVALUATION_TRAIN_YEARS = (2021, 2025)
POINT_BUCKET_CONTRACT = "polymarket_half_up_2f"
POINT_MAX_FEATURE_MISSING_FRACTION = 0.03
LIVE_POINT_MODEL_VERSION = "research_kdal_hrrr_9am_v1_live_unreleased_2026"
PROBABILITY_MODEL_VERSION = "research_kdal_hrrr_9am_v1_ordinal_pure"
PROBABILITY_FEATURE_PROFILE = "common_no_peak"
PROBABILITY_FEATURE_COUNT = 59
PROBABILITY_PROVIDERS = ('gfs', 'hrrr', 'nbm')
PROBABILITY_DEVELOPMENT_YEARS = (2023, 2024, 2025)
PROBABILITY_FORWARD_VALIDATION_YEARS = (2024, 2025)
PROBABILITY_HOLDOUT_YEAR = 2026
DATA_PROJECT_ROOT = Path('D:/dev/weather-research').resolve()
assert (DATA_PROJECT_ROOT / "data" / "processed" / "actual_highs.csv").is_file()
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS,
    _fit_feature_columns,
    _modeling_frame,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    V20_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


In [3]:
# Provider-specific HRRR source override. GFS, NBM, and observations retain
# the baseline same_day_11am_live_safe loader and 11:00-local snapshot.
import hashlib
import json
from zoneinfo import ZoneInfo

import src.calibration.station_stacking as _station_stacking

HRRR_9AM_CONTRACT = {
  "hrrr_cache_path": "D:/dev/weather-research/data/calibration/sdk_11am_hrrr_9am_cycle_v1_2021_latest/sdk_nwp_0h_cache.csv",
  "hrrr_validation_summary_path": "D:/dev/weather-research/data/calibration/sdk_11am_hrrr_9am_cycle_v1_2021_latest/validation_summary.json",
  "hrrr_data_manifest_path": "D:/dev/weather-research/data/calibration/sdk_11am_hrrr_9am_cycle_v1_2021_latest/data_manifest.json",
  "hrrr_timing_mode": "same_day_11am_hrrr_9am_cycle_v1",
  "hrrr_expected_sha256": "592acca47dea6c1d96e83e3e572eb796d7658ddafc96adf9f38a40d57fe7eebe",
  "hrrr_expected_rows": 2046,
  "hrrr_date_range": [
    "2021-01-01",
    "2026-08-08"
  ],
  "hrrr_local_timezone": "America/Chicago",
  "observation_cutoff_local": "11:00",
  "prediction_decision_time_local": "11:15"
}
HRRR_9AM_CACHE_PATH = Path(HRRR_9AM_CONTRACT["hrrr_cache_path"])
HRRR_9AM_VALIDATION_SUMMARY_PATH = Path(HRRR_9AM_CONTRACT["hrrr_validation_summary_path"])
HRRR_9AM_DATA_MANIFEST_PATH = Path(HRRR_9AM_CONTRACT["hrrr_data_manifest_path"])
HRRR_9AM_TIMING_MODE = HRRR_9AM_CONTRACT["hrrr_timing_mode"]


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _load_validated_exact_9am_hrrr():
    required_paths = (
        HRRR_9AM_CACHE_PATH,
        HRRR_9AM_VALIDATION_SUMMARY_PATH,
        HRRR_9AM_DATA_MANIFEST_PATH,
    )
    missing_paths = [str(path) for path in required_paths if not path.is_file()]
    assert not missing_paths, f"Missing exact-09:00 HRRR inputs: {missing_paths}"

    summary = json.loads(HRRR_9AM_VALIDATION_SUMMARY_PATH.read_text(encoding="utf-8"))
    manifest = json.loads(HRRR_9AM_DATA_MANIFEST_PATH.read_text(encoding="utf-8"))
    expected_rows = int(HRRR_9AM_CONTRACT["hrrr_expected_rows"])
    expected_dates = tuple(HRRR_9AM_CONTRACT["hrrr_date_range"])
    expected_sha = HRRR_9AM_CONTRACT["hrrr_expected_sha256"]
    assert summary["timing_mode"] == HRRR_9AM_TIMING_MODE
    assert summary["rows"] == summary["ok_rows"] == expected_rows
    assert summary["date_range"] == list(expected_dates)
    assert not summary["missing_dates"]
    assert not summary["issue_timestamp_error_dates"]
    assert not summary["leakage_error_dates"]
    assert not summary["forecast_window_error_dates"]
    assert manifest == {"sha256": expected_sha, "rows": expected_rows, "timing_mode": HRRR_9AM_TIMING_MODE}
    assert _sha256(HRRR_9AM_CACHE_PATH) == expected_sha

    frame = pd.read_csv(HRRR_9AM_CACHE_PATH, low_memory=False)
    assert len(frame) == expected_rows
    assert frame["contract_date"].astype(str).nunique() == expected_rows
    assert set(frame["provider"].astype(str).str.lower()) == {"hrrr"}
    assert set(frame["model"].astype(str).str.lower()) == {"hrrr"}
    assert set(frame["timing_mode"].astype(str)) == {HRRR_9AM_TIMING_MODE}
    assert set(frame["fetch_status"].astype(str).str.lower()) == {"ok"}
    assert tuple(frame["contract_date"].astype(str).agg(["min", "max"])) == expected_dates
    assert set(pd.to_numeric(frame["forecast_hour_min"], errors="raise")) == {2}
    assert set(pd.to_numeric(frame["forecast_hour_max"], errors="raise")) == {14}

    local_zone = ZoneInfo(HRRR_9AM_CONTRACT["hrrr_local_timezone"])
    issued_local = pd.to_datetime(frame["issued_at"], utc=True, errors="raise").dt.tz_convert(local_zone)
    as_of_local = pd.to_datetime(frame["forecast_as_of"], utc=True, errors="raise").dt.tz_convert(local_zone)
    contract_dates = pd.to_datetime(frame["contract_date"], errors="raise").dt.date
    assert (issued_local.dt.date == contract_dates).all()
    assert (issued_local.dt.hour == 9).all() and (issued_local.dt.minute == 0).all()
    utc_offsets = issued_local.map(lambda value: value.utcoffset().total_seconds() / 3600)
    assert set(utc_offsets) == {-6.0, -5.0}
    issue_hours_utc = pd.to_datetime(frame["issued_at"], utc=True, errors="raise").dt.hour
    assert set(issue_hours_utc) == {14, 15}
    assert (as_of_local.dt.date == contract_dates).all()
    assert (as_of_local.dt.hour == 11).all() and (as_of_local.dt.minute == 0).all()
    decision_local = pd.to_datetime(
        frame["contract_date"].astype(str) + " 11:15", errors="raise"
    ).dt.tz_localize(local_zone)
    assert (issued_local <= decision_local).all()
    assert (as_of_local <= decision_local).all()

    for column in _station_stacking.FORECAST_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
    frame = frame[_station_stacking.FORECAST_COLUMNS].copy()
    frame["source_cache_dir"] = HRRR_9AM_CACHE_PATH.parent.name
    frame["source_cache_mtime"] = HRRR_9AM_CACHE_PATH.stat().st_mtime
    return frame, summary, manifest


_baseline_forecast_loader = _station_stacking.load_same_day_provider_forecasts


def _load_forecasts_with_exact_9am_hrrr(project_root=".", timing_mode="same_day_11am", providers=("gfs", "hrrr")):
    baseline = _baseline_forecast_loader(project_root, timing_mode=timing_mode, providers=providers)
    requested = tuple(str(provider).lower() for provider in providers)
    if "hrrr" not in requested:
        return baseline
    exact_hrrr, _, _ = _load_validated_exact_9am_hrrr()
    exact_hrrr = exact_hrrr.loc[exact_hrrr["station_id"].astype(str).str.upper().eq("KDAL")].copy()
    baseline = baseline.loc[baseline["provider"].astype(str).str.lower().ne("hrrr")].copy()
    combined = pd.concat([baseline, exact_hrrr], ignore_index=True, sort=False)
    hrrr_rows = combined.loc[combined["provider"].astype(str).str.lower().eq("hrrr")]
    assert len(hrrr_rows) == int(HRRR_9AM_CONTRACT["hrrr_expected_rows"])
    assert set(hrrr_rows["timing_mode"].astype(str)) == {HRRR_9AM_TIMING_MODE}
    assert not hrrr_rows.duplicated(["station_id", "provider", "contract_date"]).any()
    return combined


_station_stacking.load_same_day_provider_forecasts = _load_forecasts_with_exact_9am_hrrr


## Exact-09:00 HRRR data readiness and chronology gate

This experiment changes only the HRRR source contract. The cache must contain
one valid KDAL row per date from 2021-01-01 through 2026-08-08, issued at exact
09:00 `America/Chicago` (14Z in CDT and 15Z in CST), using f02-f14. GFS, NBM,
Wunderground labels, and the live-safe observation snapshot remain inherited
from the KDAL V20 no-peak full-refit lineage. Observations stop at 11:00 local;
prediction and decision remain 11:15 local.


In [4]:
exact_9am_hrrr, hrrr_validation_summary, hrrr_data_manifest = _load_validated_exact_9am_hrrr()
hrrr_readiness = pd.DataFrame(
    [{
        "timing_mode": HRRR_9AM_TIMING_MODE,
        "rows": len(exact_9am_hrrr),
        "unique_dates": exact_9am_hrrr["contract_date"].nunique(),
        "first_date": exact_9am_hrrr["contract_date"].min(),
        "last_date": exact_9am_hrrr["contract_date"].max(),
        "issue_hours_utc": sorted(pd.to_datetime(exact_9am_hrrr["issued_at"], utc=True).dt.hour.unique().tolist()),
        "forecast_hour_min": int(pd.to_numeric(exact_9am_hrrr["forecast_hour_min"]).min()),
        "forecast_hour_max": int(pd.to_numeric(exact_9am_hrrr["forecast_hour_max"]).max()),
        "sha256": hrrr_data_manifest["sha256"],
        "observation_cutoff_local": HRRR_9AM_CONTRACT["observation_cutoff_local"],
        "prediction_decision_time_local": HRRR_9AM_CONTRACT["prediction_decision_time_local"],
    }]
)
hrrr_readiness


,timing_mode,rows,unique_dates,first_date,last_date,issue_hours_utc,forecast_hour_min,forecast_hour_max,sha256,observation_cutoff_local,prediction_decision_time_local
0,same_day_11am_hrrr_9am_cycle_v1,2046,2046,2021-01-01,2026-08-08,"[14, 15]",2,14,592acca47dea6c1d96e83e3e572eb796d7658ddafc96ad...,11:00,11:15


## Point-model contract

`feature_version="v11_settlement_fix_temp"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [5]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in V20_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_to_2022,2021,2021,2022
1,fold_2021_2022_to_2023,2021,2022,2023
2,fold_2021_2023_to_2024,2021,2023,2024
3,fold_2021_2024_to_2025,2021,2024,2025


In [6]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [7]:
availability = provider_availability(
    DATA_PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]

station_availability = availability.loc[availability["station_id"].eq(STATION_ID)].copy()
assert set(station_availability["provider"].astype(str)) == set(PROVIDERS), (
    "Full training readiness failed: the inherited 11 AM GFS/NBM caches and "
    "the exact-09:00 HRRR cache must all be available for KDAL."
)
observation_rows = _station_stacking.load_current_observation_features(
    DATA_PROJECT_ROOT,
    station_id=STATION_ID,
    timing_mode=TIMING_MODE,
)
assert not observation_rows.empty, (
    "Full training readiness failed: no inherited 11:00-local KDAL observation cache was found."
)

station_availability


,station_id,provider,row_count,first_contract_date,last_contract_date
4,KDAL,gfs,2021,2021-01-01,2026-07-29
5,KDAL,hrrr,2046,2021-01-01,2026-08-08
6,KDAL,nbm,2015,2021-01-01,2026-07-29


## Model Scores


In [8]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=DATA_PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v11_settlement_fix_temp",
    training_profile="v20_aligned",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    max_feature_missing_fraction=POINT_MAX_FEATURE_MISSING_FRACTION,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=V20_EXPANDING_FOLDS,
    year_split_validation_weights={2022: 1.0, 2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "experiments" / "kdal_hrrr_9am_v1",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/experiments/kdal_hrrr_9am_v1/KDAL_optuna.sqlite3')

In [9]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-08-11 12:48:47,101] A new study created in RDB with name: KDAL_v11_settlement_fix_temp_remaining_warmup_v20_aligned_base_xgboost_mae_f_wide
[I 2026-08-11 12:49:33,759] Trial 0 finished with value: 1.696413214172326 and parameters: {'n_estimators': 1342, 'learning_rate': 0.19043899115982607, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.696413214172326.
[I 2026-08-11 12:50:32,156] Trial 1 finished with value: 2.3726762977199325 and parameters: {'n_estimators': 2493, 'learning_rate': 0.001120367191095075, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 0 with value: 1.696413214172326

KeyboardInterrupt: 

## Point-model market-bucket hit rate

This score belongs to the continuous point model, not the ordinal probability
model. The configured market contract is `polymarket_half_up_2f`: two-degree Fahrenheit bracket after half-up degree rounding.
The forward score is honest chronological evidence. The holdout score is shown
separately and remains exploratory.


In [ ]:
from src.calibration.temperature_buckets import (
    point_bucket_metrics,
    point_bucket_predictions,
)
from src.calibration.v19_bucket import crossfit_ridge_predictions

point_forward_predictions = crossfit_ridge_predictions(
    result.validation_predictions,
    providers=PROBABILITY_PROVIDERS,
)
point_forward_predictions = point_forward_predictions.loc[
    point_forward_predictions["validation_year"].isin(
        PROBABILITY_FORWARD_VALIDATION_YEARS
    )
].copy()
assert not point_forward_predictions.empty
assert (
    point_forward_predictions["train_through_year"]
    < point_forward_predictions["validation_year"]
).all()

point_forward_bucket_predictions = point_bucket_predictions(
    point_forward_predictions,
    POINT_BUCKET_CONTRACT,
)
point_forward_bucket_metrics = point_bucket_metrics(
    point_forward_predictions,
    POINT_BUCKET_CONTRACT,
)
point_forward_bucket_metrics["evaluation_status"] = "honest_forward"
point_forward_bucket_metrics


In [ ]:
point_holdout_predictions = result.test_predictions.loc[
    result.test_predictions["method"].eq("ridge_stack"),
    ["contract_date", "actual_high_f", "predicted_high_f"],
].copy()
assert not point_holdout_predictions.empty

point_holdout_bucket_predictions = point_bucket_predictions(
    point_holdout_predictions,
    POINT_BUCKET_CONTRACT,
)
point_holdout_bucket_metrics = point_bucket_metrics(
    point_holdout_predictions,
    POINT_BUCKET_CONTRACT,
)
point_holdout_bucket_metrics["evaluation_status"] = "exploratory_holdout"
point_holdout_bucket_metrics


In [ ]:
point_bucket_output_dir = config.resolved_output_dir() / "point_bucket_evaluation"
point_bucket_output_dir.mkdir(parents=True, exist_ok=True)
point_forward_bucket_predictions.to_csv(
    point_bucket_output_dir / f"{STATION_ID}_forward_predictions.csv", index=False
)
point_forward_bucket_metrics.to_csv(
    point_bucket_output_dir / f"{STATION_ID}_forward_metrics.csv", index=False
)
point_holdout_bucket_predictions.to_csv(
    point_bucket_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_predictions.csv",
    index=False,
)
point_holdout_bucket_metrics.to_csv(
    point_bucket_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_metrics.csv",
    index=False,
)
point_bucket_output_dir


In [ ]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        train_years=POINT_EVALUATION_TRAIN_YEARS,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        max_feature_missing_fraction=config.effective_max_feature_missing_fraction,
        bucket_contract=POINT_BUCKET_CONTRACT,
        source_pipeline="notebooks/experiments/kdal_hrrr_9am_v1",
    )

    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this experimental notebook.")

if EXPORT_MODEL_WEIGHTS:
    import json as _point_export_json

    evaluation_point_manifest = _point_export_json.loads(
        exported_weights.manifest_path.read_text(encoding="utf-8")
    )
    assert evaluation_point_manifest["model_version"] == MODEL_VERSION
    assert evaluation_point_manifest["training"]["train_start_year"] == POINT_EVALUATION_TRAIN_YEARS[0]
    assert evaluation_point_manifest["training"]["train_end_year"] == POINT_EVALUATION_TRAIN_YEARS[1]
    assert evaluation_point_manifest["model_contract"]["max_feature_missing_fraction"] == POINT_MAX_FEATURE_MISSING_FRACTION
    assert evaluation_point_manifest["model_contract"]["bucket_contract"] == POINT_BUCKET_CONTRACT
    assert all(
        row["missing_fraction"] <= POINT_MAX_FEATURE_MISSING_FRACTION
        for row in evaluation_point_manifest["features"]["missingness_audit"]
        if row["selected"]
    )


## Optional live-production point bundle

The evaluation bundle above is frozen before the exploratory holdout and is the
only point bundle used by probability training and holdout reporting. A live
production refit may use all completed actuals, including completed holdout-year
dates, but it has a distinct version and cannot claim holdout performance as
out-of-sample evidence. Keep this export disabled until the source is committed;
then create the immutable release record in a separate promotion review.


In [ ]:
live_exported_weights = None
if EXPORT_LIVE_MODEL_WEIGHTS:
    live_exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        city_id=CITY_ID if "CITY_ID" in globals() else None,
        artifact_dir=config.resolved_output_dir(),
        model_version=LIVE_POINT_MODEL_VERSION,
        train_years=None,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        max_feature_missing_fraction=config.effective_max_feature_missing_fraction,
        bucket_contract=POINT_BUCKET_CONTRACT,
        source_pipeline="notebooks/experiments/kdal_hrrr_9am_v1",
    )
    assert live_exported_weights.bundle_path != exported_weights.bundle_path
    assert LIVE_POINT_MODEL_VERSION != MODEL_VERSION
    print(
        "Live bundle exported as an unreleased candidate. "
        "Create a clean-checkout release record before promotion."
    )
else:
    print("Live-production export disabled; evaluation bundle remains frozen.")


## Point-model feature coverage


In [ ]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


In [ ]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


## Dropped Feature Check


In [ ]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


## Morning Trend Coverage


In [ ]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


## Rounded Within 1F Accuracy


In [ ]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="holdout_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


## 2026 Exploratory Holdout Weather Brackets


In [ ]:
result.bracket_metrics


## Train-Fold 3% Missingness Audit


In [ ]:
modeling_frame, candidate_categorical, candidate_numeric = _modeling_frame(result.features, config)
candidate_features = [*candidate_categorical, *candidate_numeric]
audit_specs = [
    (fold.name, fold.train_start_year, fold.train_end_year)
    for fold in V20_EXPANDING_FOLDS
] + [("test_refit_2021_2025", 2021, 2025)]

missingness_rows = []
years = pd.to_numeric(modeling_frame["year"], errors="coerce")
for fold_name, train_start, train_end in audit_specs:
    train = modeling_frame.loc[years.between(train_start, train_end)].copy()
    retained_categorical, retained_numeric = _fit_feature_columns(
        train,
        candidate_categorical,
        candidate_numeric,
        max_missing_fraction=config.effective_max_feature_missing_fraction,
    )
    retained = set(retained_categorical) | set(retained_numeric)
    for feature in candidate_features:
        numeric_feature = feature in candidate_numeric
        values = pd.to_numeric(train[feature], errors="coerce") if numeric_feature else train[feature]
        missingness_rows.append(
            {
                "fold": fold_name,
                "train_start_year": train_start,
                "train_end_year": train_end,
                "feature": feature,
                "kind": "numeric" if numeric_feature else "categorical",
                "missing_fraction": float(values.isna().mean()),
                "retained": feature in retained,
            }
        )

fold_feature_missingness = pd.DataFrame(missingness_rows)
retained_dropped_summary = (
    fold_feature_missingness.groupby(["fold", "retained"], as_index=False)
    .agg(feature_count=("feature", "nunique"), maximum_missing_fraction=("missing_fraction", "max"))
)
fold_feature_missingness.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_feature_missingness.csv", index=False)
retained_dropped_summary, fold_feature_missingness.loc[~fold_feature_missingness["retained"]].sort_values(
    ["fold", "missing_fraction"], ascending=[True, False]
)


## Expanded 11 AM Feature Coverage and Provider Count


In [ ]:
new_feature_coverage = (
    result.features[V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
provider_count_coverage = (
    result.features["v11sf_forecast_temp_11am_provider_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("available_provider_count")
    .reset_index(name="row_count")
)
provider_count_coverage["row_pct"] = provider_count_coverage["row_count"] / len(result.features) * 100
new_feature_coverage.to_csv(config.resolved_output_dir() / f"{STATION_ID}_11am_feature_coverage.csv", index=False)
new_feature_coverage, provider_count_coverage


## New-Feature Importance


In [ ]:
new_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
new_feature_importance


## 2026 Exploratory Holdout Monthly Metrics


In [ ]:
monthly_predictions = result.test_predictions.copy()
monthly_predictions["month"] = pd.to_datetime(monthly_predictions["contract_date"], errors="coerce").dt.month
monthly_metrics = (
    monthly_predictions.dropna(subset=["month", "error_f"])
    .groupby(["method", "month"], as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        rmse_f=("error_f", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_f=("error_f", "mean"),
    )
)
monthly_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_2026_monthly_metrics.csv", index=False)
monthly_metrics


## Performance by Warm/Cool 11 AM Forecast Delta


In [ ]:
delta_by_date = result.features[[
    "contract_date",
    "v11sf_forecast_temp_11am_minus_observed_f",
]].copy()
delta_predictions = result.test_predictions.merge(delta_by_date, on="contract_date", how="left")
delta_predictions["forecast_temp_delta_bucket"] = pd.cut(
    delta_predictions["v11sf_forecast_temp_11am_minus_observed_f"],
    bins=[-np.inf, -2.0, -0.5, 0.5, 2.0, np.inf],
    labels=["cool_gt_2f", "cool_0.5_to_2f", "near_match", "warm_0.5_to_2f", "warm_gt_2f"],
)
warm_cool_metrics = (
    delta_predictions.dropna(subset=["forecast_temp_delta_bucket", "error_f"])
    .groupby(["method", "forecast_temp_delta_bucket"], observed=True, as_index=False)
    .agg(count=("error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
warm_cool_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_warm_cool_delta_metrics.csv", index=False)
warm_cool_metrics


## Ordinal Probabilities Model 2 — part of this station run

This is the station's verified **pure ordinal** setup, not a separate experiment:

- target: rounded actual degree minus rounded point-model degree;
- ordered classes: `≤-4, -3, -2, -1, 0, +1, +2, +3, ≥+4`;
- eight cumulative logistic regressions with median imputation and scaling;
- forced family `ordinal_logistic`;
- learned-model blend weight `1.0` (no empirical probability blend);
- development years: `[2023, 2024, 2025]`;
- honest forward-validation years: `[2024, 2025]`, each trained only on
  earlier development years;
- the last 90 days of each outer training period tune regularization,
  class weighting, and temperature;
- final probability artifact fitted on all available development rows;
- 2026 is an exploratory holdout,
  **not** an out-of-fold training fold.

Preprocessing is fitted independently inside each chronological training fold.
Continuous features are standardized after median imputation. The verified
implementation does not apply skew transforms or feature
winsorization/clipping.

Degree probabilities are aggregated into the actual 2°F market buckets after
prediction, so bucket boundaries do not need to be centered on the point degree.
The ordinal output remains shadow/research-only until fresh promotion gates pass.


In [ ]:
import json

from src.calibration.bucket_probability import (
    build_probability_frame,
    default_candidate_specs,
    evaluate_probability_holdout,
    export_probability_bundle,
    fit_probability_system,
    probability_feature_names,
    probability_metrics,
    sha256_file,
)
from src.calibration.v19_bucket import crossfit_ridge_predictions


### Exact ordinal-regression feature contract


In [ ]:
ordinal_feature_contract = pd.DataFrame(
    {
        "position": range(
            1,
            len(
                probability_feature_names(
                    include_peak_features=False,
                    feature_profile=PROBABILITY_FEATURE_PROFILE,
                )
            )
            + 1,
        ),
        "feature": probability_feature_names(
            include_peak_features=False,
            feature_profile=PROBABILITY_FEATURE_PROFILE,
        ),
    }
)
ordinal_feature_contract


### Fit with chronological [2024, 2025] outer validation


In [ ]:
point_forward_predictions = crossfit_ridge_predictions(
    result.validation_predictions,
    providers=PROBABILITY_PROVIDERS,
)
assert not point_forward_predictions.empty
assert (
    point_forward_predictions["train_through_year"]
    < point_forward_predictions["validation_year"]
).all()

ordinal_training_frame = build_probability_frame(
    result.features,
    point_forward_predictions,
    result.validation_predictions,
    include_peak_features=False,
    feature_profile=PROBABILITY_FEATURE_PROFILE,
)
ordinal_candidate_specs = [
    spec
    for spec in default_candidate_specs()
    if spec.family in {"empirical", "ordinal_logistic"}
]
ordinal_bundle, ordinal_forward_predictions, ordinal_tuning = (
    fit_probability_system(
        ordinal_training_frame,
        station_id=STATION_ID,
        point_model_version=MODEL_VERSION,
        point_bundle_sha256=sha256_file(exported_weights.bundle_path),
        include_peak_features=False,
        feature_profile=PROBABILITY_FEATURE_PROFILE,
        model_version=PROBABILITY_MODEL_VERSION,
        candidate_specs=ordinal_candidate_specs,
        forced_family="ordinal_logistic",
        blend_weights=(1.0,),
        development_years=PROBABILITY_DEVELOPMENT_YEARS,
        forward_validation_years=PROBABILITY_FORWARD_VALIDATION_YEARS,
    )
)
assert ordinal_bundle["selected_family"] == "ordinal_logistic"
assert ordinal_bundle["blend_weight"] == 1.0
probability_metrics(ordinal_forward_predictions)


### Verify chronology and the frozen probability contract


In [ ]:
ordinal_forward_dates = pd.to_datetime(
    ordinal_forward_predictions["contract_date"]
)
assert set(ordinal_forward_predictions["validation_year"]) == set(
    PROBABILITY_FORWARD_VALIDATION_YEARS
)
assert (
    pd.to_datetime(ordinal_forward_predictions["model_training_cutoff"])
    < ordinal_forward_dates
).all()
assert (
    pd.to_datetime(
        ordinal_forward_predictions["calibration_training_cutoff"]
    )
    < pd.to_datetime(
        ordinal_forward_predictions["calibration_validation_start"]
    )
).all()
assert (
    pd.to_datetime(
        ordinal_forward_predictions["calibration_validation_cutoff"]
    )
    < ordinal_forward_dates
).all()
assert ordinal_bundle["selected_family"] == "ordinal_logistic"
assert ordinal_bundle["family_selection_mode"] == "forced"
assert ordinal_bundle["blend_weight"] == 1.0
assert ordinal_bundle["feature_profile"] == PROBABILITY_FEATURE_PROFILE
assert len(ordinal_bundle["feature_names"]) == PROBABILITY_FEATURE_COUNT
assert not any("peak" in name.lower() for name in ordinal_bundle["feature_names"])
{
    "development_years": list(PROBABILITY_DEVELOPMENT_YEARS),
    "forward_validation_years": sorted(
        ordinal_forward_predictions["validation_year"].unique().tolist()
    ),
    "final_training_start": ordinal_bundle["training_start"],
    "final_training_cutoff": ordinal_bundle["training_cutoff"],
}


### Evaluate the frozen ordinal model on the 2026 holdout


In [ ]:
holdout_point_predictions = result.test_predictions.loc[
    result.test_predictions["method"].eq("ridge_stack"),
    ["contract_date", "actual_high_f", "predicted_high_f"],
].copy()
assert not holdout_point_predictions.empty

ordinal_holdout_predictions, ordinal_holdout_metrics = (
    evaluate_probability_holdout(
        result.features,
        holdout_point_predictions,
        result.test_predictions,
        ordinal_bundle,
        holdout_year=PROBABILITY_HOLDOUT_YEAR,
    )
)
assert not ordinal_holdout_predictions.empty
assert not ordinal_holdout_metrics.empty
for probability_column in (
    "offset_probabilities",
    "degree_probabilities",
    "bucket_probabilities",
):
    assert ordinal_holdout_predictions[probability_column].map(
        lambda probabilities: np.isclose(
            sum(float(value) for value in probabilities.values()),
            1.0,
            atol=1e-10,
        )
    ).all()

ordinal_bundle["holdout_metrics"] = ordinal_holdout_metrics.iloc[0].to_dict()
ordinal_bundle["holdout_status"] = "exploratory"
ordinal_bundle["historical_acceptance"] = {
    "passed": False,
    "reasons": ["fresh_shadow_data_required"],
    "holdout_status": "exploratory_previously_inspected",
}
ordinal_holdout_metrics


### Export point and probability outputs together


In [ ]:
probability_output_dir = config.resolved_output_dir() / "ordinal_probability"
probability_output_dir.mkdir(parents=True, exist_ok=True)

serializable_forward = ordinal_forward_predictions.copy()
serializable_forward["offset_probabilities"] = (
    serializable_forward["offset_probabilities"].map(
        lambda value: json.dumps(value, sort_keys=True)
    )
)
serializable_forward.to_csv(
    probability_output_dir / f"{STATION_ID}_forward_probability_predictions.csv",
    index=False,
)
ordinal_tuning.to_csv(
    probability_output_dir / f"{STATION_ID}_probability_tuning.csv",
    index=False,
)
probability_metrics(ordinal_forward_predictions).to_csv(
    probability_output_dir / f"{STATION_ID}_forward_probability_metrics.csv",
    index=False,
)

serializable_holdout = ordinal_holdout_predictions.copy()
for column in (
    "offset_probabilities",
    "degree_probabilities",
    "bucket_probabilities",
):
    serializable_holdout[column] = serializable_holdout[column].map(
        lambda value: json.dumps(value, sort_keys=True)
    )
serializable_holdout.to_csv(
    probability_output_dir
    / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_probability_holdout_predictions.csv",
    index=False,
)
ordinal_holdout_metrics.to_csv(
    probability_output_dir
    / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_probability_holdout_metrics.csv",
    index=False,
)
ordinal_feature_contract.to_csv(
    probability_output_dir / f"{STATION_ID}_ordinal_feature_contract.csv",
    index=False,
)

ordinal_bundle_path, ordinal_manifest_path = export_probability_bundle(
    ordinal_bundle,
    probability_output_dir / "model_weights",
    source_identity={
        "pipeline": "station_training_baseline",
        "notebook": "notebooks/experiments/kdal_hrrr_9am_v1/train_KDAL.ipynb",
        "point_workflow": "KDAL V20 no-peak full-refit lineage with exact-09:00-local HRRR",
        "probability_setup": "Ordinal Probabilities Model 2",
    },
)
ordinal_manifest = json.loads(ordinal_manifest_path.read_text(encoding="utf-8"))
assert ordinal_manifest["point_bundle_sha256"] == sha256_file(
    exported_weights.bundle_path
)
assert ordinal_manifest["artifact_integrity"]["bundle_sha256"] == sha256_file(
    ordinal_bundle_path
)
{
    "point_bundle": exported_weights.bundle_path,
    "point_manifest": exported_weights.manifest_path,
    "ordinal_bundle": ordinal_bundle_path,
    "ordinal_manifest": ordinal_manifest_path,
    "output_dir": probability_output_dir,
}
